In [21]:
#Imports
import os
import numpy as np
import pandas as pd
import tkinter as tk
from tkinter import filedialog
import glob
import calcFunctions as calcFunc
import inputsOutputsFunctions as inpOutsFunc
from pathlib import Path
from tqdm import tqdm

In [22]:
def select_root_path()->str:
    """Opens a single dialog to select the top-level ROOT_PATH."""
    root_path = filedialog.askdirectory(title='Select the Top-Level ROOT_PATH Folder')
    return root_path

In [23]:
# Create Tk root
root = tk.Tk()
# Hide the main window
root.withdraw()
root.call('wm', 'attributes', '.', '-topmost', True)
ROOT_PATH = select_root_path()

In [ ]:
#Specific grain boundary function
def grain_boundaries(GB_file_path:str, GI_file_path:str,save_path:str, pixel_index:int, GB_colour:str, GI_colour:str, datatype:str)->float:
    """
    Fits Gaussian to GB and GI data, plots the results, saves the plot, 
    and writes the analysis data to a text file.
    """
    # Pixel number corresponds to the 'pixels' value (1 to 5)
    pixels = int(pixel_index+1)

    # Perform fitting
    GB_x_raw_1, GB_y_raw_1, GB_x_1, GB_y_1, GB_peak_1, GB_fwhm_1 = calcFunc.fit_gaussian(inpOutsFunc.loadtxtFile(GB_file_path))
    GI_x_raw_1, GI_y_raw_1, GI_x_1, GI_y_1, GI_peak_1, GI_fwhm_1 = calcFunc.fit_gaussian(inpOutsFunc.loadtxtFile(GI_file_path))
    peak_diff_1 = (GB_peak_1 - GI_peak_1)*1000

    inpOutsFunc.storeAnalysis(GB_peak_1,GB_fwhm_1,0,0,GI_peak_1,GI_fwhm_1,0,0,peak_diff_1,save_path,pixels,scaleFactor=1000)
    
    # #Adjusting arrays to give results in mV
    GB_x_raw = np.round(GB_x_raw_1*1000,4)
    GI_x_raw = np.round(GI_x_raw_1*1000,4)
    GB_x_fit= np.round(GB_x_1*1000,4)
    GI_x_fit= np.round(GI_x_1*1000,4)

    # # Plotting functions: 
    inpOutsFunc.generateFigures(GB_x_raw,GB_x_fit,GI_x_raw,GI_x_fit,GB_y_raw_1,GB_y_1,GI_y_raw_1,GI_y_1,GB_colour,GI_colour,save_path,datatype,pixels)            
    return peak_diff_1

In [25]:
Colours={"unpassivated":{'colour1': "#5CBBFF", 'colour2': "#003357"},
         "passivated":{'colour1': "#D69AAB", 'colour2': "#8D1730"}}

In [26]:
filetype = ".txt"
directories=[]
for i in glob.glob(f"{ROOT_PATH}/**/*{filetype}",recursive=True):
    filePath = Path(i)
    #skips the directory named Plots
    if filePath.parent.parent.parent.name=='Plots': continue
    directories.append(filePath.parent)
directories = sorted(set(directories))

In [ ]:
# Initialize dictionary to store results
all_results = {}
for i in tqdm(range(0,len(directories)-1,2)):
    if(len(glob.glob(f"{directories[i]}/*.txt"))!=len(glob.glob(f"{directories[i+1]}/*.txt"))):
        raise ValueError("Different number of GI and GB files")
    parent = directories[i].parent.parent.name
    dataType = directories[i].parent.name
    measureType = directories[i].name

    save_path = os.path.join(ROOT_PATH, 'Plots', parent, dataType)
    os.makedirs(save_path, exist_ok=True)
    
    y_peaks_adjusted = []
    
    for GB_file, GI_file in zip(sorted(glob.glob(f"{directories[i]}/*{filetype}")),sorted(glob.glob(f"{directories[i+1]}/*{filetype}"))):
        # use file name to find the index
        index = int(Path(GB_file).name.split('.')[0])
        y_peaks_adjusted.append(round(grain_boundaries(GB_file,GI_file,save_path,index,Colours[parent.lower()]["colour1"],Colours[parent.lower()]["colour2"],measureType),4))

    all_results[f"{parent}_{dataType}"]=y_peaks_adjusted

100%|██████████| 10/10 [07:55<00:00, 47.56s/it]


---Summary Plots---

In [28]:
FINAL_PLOTS_SAVE_PATH = filedialog.askdirectory(title='Select Folder to Save All Final Summary Plots')
all_results = pd.DataFrame(all_results)

In [ ]:
inpOutsFunc.CPDPlot(*np.split(all_results[["passivated_Dark_CPD","passivated_Illuminated_CPD","passivated_Recovery_CPD"]].to_numpy(),[1,2],axis=1),FINAL_PLOTS_SAVE_PATH,"passivated")
inpOutsFunc.SPVPlot(*np.split(all_results[["passivated_Illuminated_SPV","passivated_Recovery_SPV"]].to_numpy(),[1],axis=1),FINAL_PLOTS_SAVE_PATH,"passivated")

In [ ]:
inpOutsFunc.CPDPlot(*np.split(all_results[["unpassivated_Dark_CPD","unpassivated_Illuminated_CPD","unpassivated_Recovery_CPD"]].to_numpy(),[1,2],axis=1),FINAL_PLOTS_SAVE_PATH,"unpassivated")
inpOutsFunc.SPVPlot(*np.split(all_results[["unpassivated_Illuminated_SPV","unpassivated_Recovery_SPV"]].to_numpy(),[1],axis=1),FINAL_PLOTS_SAVE_PATH,"unpassivated")